# Documentação API
https://info.dengue.mat.br/services/api/doc

# **Análises e Discussões**
## **Como analisar dados sobre a dengue?**
Fonte: https://basedosdados.org/search

Podemos avaliar os casos de dengue em números absolutos ou de forma comparativa. A forma padrão de comparar cidades, estados e países em estudos epidemiológicos é a partir da taxa de incidência, que calcula os casos da doença para cada 100 mil habitantes. Outras análises possíveis incluem avaliar a incidência por características socioeconômicas da população, como idade, sexo e raça/cor. E como as fichas incluem informações da evolução da doença em casos de óbito, podemos estimar a mortalidade geral da doença na população e a taxa de letalidade da dengue (número de óbitos em um período de tempo sobre o número de pessoas doentes pelo agravo).

## **Como a dengue evoluiu no Brasil?**

O número de notificações de dengue em 2024 aumentou cerca de 350% em relação a 2023. Considerando as primeiras cinco semanas epidemiológicas do ano, o número de casos aumentou cerca de 465% com relação a 2023, mas diminuiu cerca de 57% com relação a 2025. Em números absolutos, a cidade de São Paulo registrou maior número de notificações de dengue em 2024, com cerca de 652.225 notificações registradas ao longo do ano. Belo Horizonte, Brasília, Campinas e Rio de Janeiro são outros destaques quanto ao elevado número de casos prováveis de dengue, entre 100 e 230 mil casos.  Em 2025, a cidade que está liderando o ranking no momento é São José do Rio Preto, com 24.775 notificações. São Paulo e São José do Rio Preto também ocuparam o primeiro lugar no ranking de óbitos para os dois anos, respectivamente.

# Bibliotecas

In [2]:
import pandas as pd
import requests

# Dicionários

In [3]:
# Dicionário IBGE
ibge = pd.read_excel('/content/drive/MyDrive/Portfólio/Dengue/RELATORIO_DTB_BRASIL_SUBDISTRITO.xls', header=6)
ibge.head()

,UF,Nome_UF,Região Geográfica Intermediária,Nome Região Geográfica Intermediária,Região Geográfica Imediata,Nome Região Geográfica Imediata,Município,Código Município Completo,Nome_Município,Distrito,Código de Distrito Completo,Nome_Distrito,Subdistrito,Código de Subdistrito Completo,Nome_Subdistrito
0,11,Rondônia,1101,Porto Velho,110001,Porto Velho,205,1100205,Porto Velho,5,110020505,Porto Velho,6,11002050506,Zona 01
1,11,Rondônia,1101,Porto Velho,110001,Porto Velho,205,1100205,Porto Velho,5,110020505,Porto Velho,7,11002050507,Zona 02
2,11,Rondônia,1101,Porto Velho,110001,Porto Velho,205,1100205,Porto Velho,5,110020505,Porto Velho,8,11002050508,Zona 03
3,11,Rondônia,1101,Porto Velho,110001,Porto Velho,205,1100205,Porto Velho,5,110020505,Porto Velho,9,11002050509,Zona 04
4,11,Rondônia,1101,Porto Velho,110001,Porto Velho,205,1100205,Porto Velho,5,110020505,Porto Velho,10,11002050510,Zona 05


In [4]:
ibge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 645 entries, 0 to 644
Data columns (total 15 columns):
 #   Column                                Non-Null Count  Dtype 
---  ------                                --------------  ----- 
 0   UF                                    645 non-null    int64 
 1   Nome_UF                               645 non-null    object
 2   Região Geográfica Intermediária       645 non-null    int64 
 3   Nome Região Geográfica Intermediária  645 non-null    object
 4   Região Geográfica Imediata            645 non-null    int64 
 5   Nome Região Geográfica Imediata       645 non-null    object
 6   Município                             645 non-null    int64 
 7   Código Município Completo             645 non-null    int64 
 8   Nome_Município                        645 non-null    object
 9   Distrito                              645 non-null    int64 
 10  Código de Distrito Completo           645 non-null    int64 
 11  Nome_Distrito                   

# Extração

In [22]:
# URL base da API
url = "https://info.dengue.mat.br/api/alertcity"

# Parâmetros fixos
params_base = {
    "disease": "dengue",
    "format": "csv",
    "ew_start": 1,
    "ew_end": 53,
    "ey_start": 2010,
    "ey_end": 2025,
}

# Lista de códigos de municípios (IBGE)
geocode_list = ibge['Código Município Completo'].unique()

# Dicionários Municípios IBGE
mapa_nome_municipio = dict(zip(muni_ibge['Código Município Completo'], muni_ibge['Nome_Município']))
mapa_uf = dict(zip(muni_ibge['Código Município Completo'], muni_ibge['Nome_UF']))

# Colunas a renomear
colunas_renomear = {
    "nivel": "nivel_alerta",
    "pop": "pop_est_ibge"
    }

# Colunas a excluir
colunas_excluir = ['Localidade_id',
                   'versao_modelo',
                   'tweet',
                   'id',
                   'casprov_est_min',
                   'casprov_est_max',
                   'casprov_est',
                   'casconf'
                   ]

# Lista para armazenar os DataFrames
lista_dfs = []

# Loop pelos municípios
for muni in geocode_list:
    params = params_base.copy()
    params["geocode"] = muni

    try:
        # Montar URL com parâmetros
        full_url = requests.Request('GET', url, params=params).prepare().url
        print(f"✅ URL formada: {full_url}, baixando dados!")

        # Ler os dados diretamente do link CSV
        df = pd.read_csv(full_url)
        df["cod_mun"] = muni
        print("✅ Dados lidos e baixados!")

        # Renomear colunas
        df.rename(columns=colunas_renomear, inplace=True)

        # Excluir colunas
        df.drop(columns=[col for col in colunas_excluir if col in df.columns], inplace=True)

        # Criar colunas com nome do município e UF
        df["nom_mun"] = df["cod_mun"].map(mapa_nome_municipio)
        df["nom_uf"] = df["cod_mun"].map(mapa_uf)

        # Adiciona o df à lista
        lista_dfs.append(df)
        print("✅ Dados lidos e processados!")

    except Exception as e:
        print(f"❌ Erro ao extrair dados para o município: {muni}")
        print("   →", e)

# Junta todos os DataFrames em um só
dados_completos = pd.concat(lista_dfs, ignore_index=True)

# Exemplo de visualização ou exportação
print("✅ Dados extraídos com sucesso!")

# Salvando
dados_completos.to_csv("/content/drive/MyDrive/Portfólio/Dengue/dados_dengue_completos.csv", index=False)
print("✅ Dados salvos com sucesso!")

✅ URL formada: https://info.dengue.mat.br/api/alertcity?disease=dengue&format=csv&ew_start=1&ew_end=53&ey_start=2010&ey_end=2025&geocode=1100205, baixando dados!
✅ Dados lidos e baixados!
✅ Dados lidos e processados!
✅ URL formada: https://info.dengue.mat.br/api/alertcity?disease=dengue&format=csv&ew_start=1&ew_end=53&ey_start=2010&ey_end=2025&geocode=1302603, baixando dados!
✅ Dados lidos e baixados!
✅ Dados lidos e processados!
✅ URL formada: https://info.dengue.mat.br/api/alertcity?disease=dengue&format=csv&ew_start=1&ew_end=53&ey_start=2010&ey_end=2025&geocode=2211001, baixando dados!
✅ Dados lidos e baixados!
✅ Dados lidos e processados!
✅ URL formada: https://info.dengue.mat.br/api/alertcity?disease=dengue&format=csv&ew_start=1&ew_end=53&ey_start=2010&ey_end=2025&geocode=2304400, baixando dados!
✅ Dados lidos e baixados!
✅ Dados lidos e processados!
✅ URL formada: https://info.dengue.mat.br/api/alertcity?disease=dengue&format=csv&ew_start=1&ew_end=53&ey_start=2010&ey_end=2025&geo

# Visualizando

In [23]:
dados_completos.head()

,data_iniSE,SE,casos_est,casos_est_min,casos_est_max,casos,p_rt1,p_inc100k,nivel_alerta,Rt,...,nivel_inc,umidmed,umidmin,tempmed,tempmax,casprov,notif_accum_year,cod_mun,nom_mun,nom_uf
0,2025-03-30,202514,66.0,13,227.0,0,0.722749,14.293510,2,1.111018,...,2,92.454817,82.719567,25.296450,27.888067,0.0,27775,1100205,Porto Velho,Rondônia
1,2025-03-23,202513,53.0,12,191.0,0,0.272949,11.478123,2,0.892608,...,1,92.602114,83.153757,25.342657,27.698114,0.0,27775,1100205,Porto Velho,Rondônia
2,2025-03-16,202512,62.0,31,149.0,23,0.693847,13.427238,2,1.097276,...,1,88.120071,69.496243,26.350014,30.176657,13.0,27775,1100205,Porto Velho,Rondônia
3,2025-03-09,202511,62.0,41,122.0,35,0.849247,13.427238,2,1.214188,...,1,91.380114,75.729743,25.271143,28.708043,8.0,27775,1100205,Porto Velho,Rondônia
4,2025-03-02,202510,56.0,40,96.0,36,0.796111,12.127828,2,1.176168,...,1,93.840186,83.920229,24.744886,26.905200,9.0,27775,1100205,Porto Velho,Rondônia


In [24]:
# Informações Gerais
dados_completos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42984 entries, 0 to 42983
Data columns (total 25 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   data_iniSE        42984 non-null  object 
 1   SE                42984 non-null  int64  
 2   casos_est         42984 non-null  float64
 3   casos_est_min     42984 non-null  int64  
 4   casos_est_max     42924 non-null  float64
 5   casos             42984 non-null  int64  
 6   p_rt1             42984 non-null  float64
 7   p_inc100k         42984 non-null  float64
 8   nivel_alerta      42984 non-null  int64  
 9   Rt                42984 non-null  float64
 10  pop_est_ibge      42984 non-null  float64
 11  tempmin           42268 non-null  float64
 12  umidmax           40812 non-null  float64
 13  receptivo         42410 non-null  float64
 14  transmissao       42906 non-null  float64
 15  nivel_inc         42984 non-null  int64  
 16  umidmed           40670 non-null  float6

In [29]:
# Estatísticas Descritivas
descritivas = dados_completos.describe().T
pd.options.display.float_format = '{:.2f}'.format
descritivas

,count,mean,std,min,25%,50%,75%,max
SE,42984.00,201740.30,440.35,201001.00,201343.75,201733.50,202123.25,202514.00
casos_est,42984.00,154.08,803.51,0.00,0.00,9.00,73.00,30283.00
casos_est_min,42984.00,153.74,803.31,0.00,0.00,9.00,72.00,30283.00
casos_est_max,42924.00,132.10,478.48,0.00,0.00,9.00,73.00,9885.00
casos,42984.00,153.57,803.23,0.00,0.00,9.00,72.00,30283.00
p_rt1,42984.00,0.39,0.38,0.00,0.00,0.32,0.77,1.00
p_inc100k,42984.00,19.71,71.80,0.00,0.00,2.42,10.95,1605.70
nivel_alerta,42984.00,1.58,1.02,1.00,1.00,1.00,2.00,4.00
Rt,42984.00,1.17,2.40,0.00,0.00,0.81,1.23,18.52
pop_est_ibge,42984.00,739557.57,1126907.87,5015.00,97334.00,360086.00,823302.00,6747815.00


In [30]:
# Verificando valores ausentes
dados_completos.isnull().sum()

,0
data_iniSE,0
SE,0
casos_est,0
casos_est_min,0
casos_est_max,60
casos,0
p_rt1,0
p_inc100k,0
nivel_alerta,0
Rt,0


In [31]:
# Visualizando valores categóricos
for col in dados_completos.select_dtypes(include='object').columns:
    print(f'Contagem de valores para {col}:')
    print(dados_completos[col].value_counts())
    print('-'*42)

Contagem de valores para data_iniSE:
data_iniSE
2010-01-03    54
2025-03-30    54
2025-03-23    54
2025-03-16    54
2025-03-09    54
              ..
2024-12-08    54
2024-12-15    54
2024-12-22    54
2024-12-29    54
2025-01-05    54
Name: count, Length: 796, dtype: int64
------------------------------------------
Contagem de valores para nom_mun:
nom_mun
Porto Velho              796
Manaus                   796
Teresina                 796
Fortaleza                796
Natal                    796
Camaragibe               796
Petrolina                796
Recife                   796
Maceió                   796
Feira de Santana         796
Salvador                 796
Barra Longa              796
Belo Horizonte           796
Betim                    796
Contagem                 796
Cordisburgo              796
Itajubá                  796
Juiz de Fora             796
Ponte Nova               796
Sete Lagoas              796
Vila Velha               796
Vitória                  796
Bar